# Notebook 5 – Feature Engineering and Preprocessing

## 1. Import Libraries

In [23]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import joblib
import os

## 2. Loading the Dataset Splits

In [24]:
train = pd.read_csv("../data/artifacts/train.csv")
validation = pd.read_csv("../data/artifacts/validation.csv")
test = pd.read_csv("../data/artifacts/test.csv")

print("Train shape:", train.shape)
print("Validation shape:", validation.shape)
print("Test shape:", test.shape)

Train shape: (67533, 21)
Validation shape: (14471, 21)
Test shape: (14472, 21)


## 3. Geographic Feature Engineering

In [25]:
# Load aggregated geographic features created in Notebook 4

geo_features = pd.read_csv(
    "../data/artifacts/geo_features.csv"
)

print("Geographic features shape:", geo_features.shape)
print(geo_features.head())

Geographic features shape: (19015, 3)
   geolocation_zip_code_prefix  geolocation_lat  geolocation_lng
0                         1001       -23.550190       -46.634024
1                         1002       -23.548146       -46.634979
2                         1003       -23.548994       -46.635731
3                         1004       -23.549799       -46.634757
4                         1005       -23.549456       -46.636733


In [26]:
print(
    "Duplicate ZIP prefixes:",
    geo_features["geolocation_zip_code_prefix"].duplicated().sum()
)

Duplicate ZIP prefixes: 0


In [27]:
# Merge geographic coordinates into all dataset splits

train = train.merge(
    geo_features,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

validation = validation.merge(
    geo_features,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

test = test.merge(
    geo_features,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

In [28]:
# Remove the duplicate geographic ZIP key after merging

for df in [train, validation, test]:
    df.drop(
        columns=["geolocation_zip_code_prefix"],
        inplace=True
    )

In [29]:
print("Train shape:", train.shape)
print("Validation shape:", validation.shape)
print("Test shape:", test.shape)

Train shape: (67533, 23)
Validation shape: (14471, 23)
Test shape: (14472, 23)


In [30]:
geo_columns = [
    "geolocation_lat",
    "geolocation_lng"
]

print("Missing geographic values in train:")
print(train[geo_columns].isna().sum())

print("\nMissing geographic values in validation:")
print(validation[geo_columns].isna().sum())

print("\nMissing geographic values in test:")
print(test[geo_columns].isna().sum())

Missing geographic values in train:
geolocation_lat    181
geolocation_lng    181
dtype: int64

Missing geographic values in validation:
geolocation_lat    40
geolocation_lng    40
dtype: int64

Missing geographic values in test:
geolocation_lat    43
geolocation_lng    43
dtype: int64


## 4. Time-Based Feature Engineering

In [31]:
def create_time_features(df):
    df = df.copy()

    df["order_purchase_timestamp"] = pd.to_datetime(
        df["order_purchase_timestamp"]
    )

    df["purchase_year"] = (
        df["order_purchase_timestamp"].dt.year
    )

    df["purchase_month"] = (
        df["order_purchase_timestamp"].dt.month
    )

    df["purchase_dayofweek"] = (
        df["order_purchase_timestamp"].dt.dayofweek
    )

    df["purchase_hour"] = (
        df["order_purchase_timestamp"].dt.hour
    )

    return df

In [32]:
train = create_time_features(train)
validation = create_time_features(validation)
test = create_time_features(test)

In [33]:
time_features = [
    "purchase_year",
    "purchase_month",
    "purchase_dayofweek",
    "purchase_hour"
]

train[time_features].head()

,purchase_year,purchase_month,purchase_dayofweek,purchase_hour
0,2016,9,3,12
1,2016,10,0,9
2,2016,10,0,16
3,2016,10,0,21
4,2016,10,0,21


## 5. Feature Selection

In [34]:
numerical_features = [
    "total_items",
    "total_price",
    "total_freight",
    "unique_products",
    "unique_sellers",
    "total_payments",
    "payment_count",
    "payment_types",
    "purchase_year",
    "purchase_month",
    "purchase_dayofweek",
    "purchase_hour",
    "geolocation_lat",
    "geolocation_lng"
]

categorical_features = [
    "customer_state"
]

target = "late"

## 6. Removing Unnecessary Columns

In [35]:
drop_columns = [
    "late",
    "order_id",
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_zip_code_prefix",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "order_status"
]

## 7. Separating Features and Target

In [36]:
X_train = train.drop(columns=drop_columns)
y_train = train[target]

X_validation = validation.drop(columns=drop_columns)
y_validation = validation[target]

X_test = test.drop(columns=drop_columns)
y_test = test[target]

print("X_train shape:", X_train.shape)
print("X_validation shape:", X_validation.shape)
print("X_test shape:", X_test.shape)

print("\nFeatures:")
print(X_train.columns.tolist())

X_train shape: (67533, 15)
X_validation shape: (14471, 15)
X_test shape: (14472, 15)

Features:
['customer_state', 'total_items', 'total_price', 'total_freight', 'unique_products', 'unique_sellers', 'total_payments', 'payment_count', 'payment_types', 'geolocation_lat', 'geolocation_lng', 'purchase_year', 'purchase_month', 'purchase_dayofweek', 'purchase_hour']


In [37]:
print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

print("\nFeatures used by X_train:")
print(X_train.columns.tolist())

Numerical features:
['total_items', 'total_price', 'total_freight', 'unique_products', 'unique_sellers', 'total_payments', 'payment_count', 'payment_types', 'purchase_year', 'purchase_month', 'purchase_dayofweek', 'purchase_hour', 'geolocation_lat', 'geolocation_lng']

Categorical features:
['customer_state']

Features used by X_train:
['customer_state', 'total_items', 'total_price', 'total_freight', 'unique_products', 'unique_sellers', 'total_payments', 'payment_count', 'payment_types', 'geolocation_lat', 'geolocation_lng', 'purchase_year', 'purchase_month', 'purchase_dayofweek', 'purchase_hour']


## 8. Preprocessing Pipeline

In [38]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## 9. Fitting and Applying the Preprocessing Pipeline

In [39]:
X_train_processed = preprocessor.fit_transform(X_train)

X_validation_processed = preprocessor.transform(X_validation)

X_test_processed = preprocessor.transform(X_test)

print("Original X_train shape:", X_train.shape)
print("Processed X_train shape:", X_train_processed.shape)

print("Original X_validation shape:", X_validation.shape)
print("Processed X_validation shape:", X_validation_processed.shape)

print("Original X_test shape:", X_test.shape)
print("Processed X_test shape:", X_test_processed.shape)

Original X_train shape: (67533, 15)
Processed X_train shape: (67533, 41)
Original X_validation shape: (14471, 15)
Processed X_validation shape: (14471, 41)
Original X_test shape: (14472, 15)
Processed X_test shape: (14472, 41)


In [40]:
print(
    "Missing values in processed train:",
    np.isnan(X_train_processed).sum()
)

print(
    "Missing values in processed validation:",
    np.isnan(X_validation_processed).sum()
)

print(
    "Missing values in processed test:",
    np.isnan(X_test_processed).sum()
)

Missing values in processed train: 0
Missing values in processed validation: 0
Missing values in processed test: 0


## 10. Final Feature Names

In [41]:
feature_names = preprocessor.get_feature_names_out()

print("Number of final features:", len(feature_names))

print("\nFirst 20 features:")
print(feature_names[:20])

Number of final features: 41

First 20 features:
['num__total_items' 'num__total_price' 'num__total_freight'
 'num__unique_products' 'num__unique_sellers' 'num__total_payments'
 'num__payment_count' 'num__payment_types' 'num__purchase_year'
 'num__purchase_month' 'num__purchase_dayofweek' 'num__purchase_hour'
 'num__geolocation_lat' 'num__geolocation_lng' 'cat__customer_state_AC'
 'cat__customer_state_AL' 'cat__customer_state_AM'
 'cat__customer_state_AP' 'cat__customer_state_BA'
 'cat__customer_state_CE']


In [42]:
feature_list = pd.DataFrame({
    "feature": feature_names
})

feature_list.head(20)

,feature
0,num__total_items
1,num__total_price
2,num__total_freight
3,num__unique_products
4,num__unique_sellers
5,num__total_payments
6,num__payment_count
7,num__payment_types
8,num__purchase_year
9,num__purchase_month


## 11. Saving Reusable Artifacts

In [43]:
os.makedirs("../data/artifacts", exist_ok=True)

feature_list.to_csv(
    "../data/artifacts/feature_list.csv",
    index=False
)

print("Feature list saved.")

joblib.dump(
    preprocessor,
    "../data/artifacts/preprocessor.joblib"
)

print("Preprocessor saved.")

np.save(
    "../data/artifacts/X_train_processed.npy",
    X_train_processed
)

np.save(
    "../data/artifacts/X_validation_processed.npy",
    X_validation_processed
)

np.save(
    "../data/artifacts/X_test_processed.npy",
    X_test_processed
)

print("Processed feature tables saved.")

np.save(
    "../data/artifacts/y_train.npy",
    y_train.to_numpy()
)

np.save(
    "../data/artifacts/y_validation.npy",
    y_validation.to_numpy()
)

np.save(
    "../data/artifacts/y_test.npy",
    y_test.to_numpy()
)

print("Target arrays saved.")

Feature list saved.
Preprocessor saved.
Processed feature tables saved.
Target arrays saved.


## 12. Final Verification

In [44]:
print("FINAL FEATURE ENGINEERING CHECK")
print("--------------------------------")

print("X_train_processed:", X_train_processed.shape)
print("X_validation_processed:", X_validation_processed.shape)
print("X_test_processed:", X_test_processed.shape)

print("Number of features:", len(feature_names))

print(
    "Missing values in train:",
    np.isnan(X_train_processed).sum()
)

print(
    "Missing values in validation:",
    np.isnan(X_validation_processed).sum()
)

print(
    "Missing values in test:",
    np.isnan(X_test_processed).sum()
)

print("\nArtifacts saved successfully.")

FINAL FEATURE ENGINEERING CHECK
--------------------------------
X_train_processed: (67533, 41)
X_validation_processed: (14471, 41)
X_test_processed: (14472, 41)
Number of features: 41
Missing values in train: 0
Missing values in validation: 0
Missing values in test: 0

Artifacts saved successfully.
